# Experiment: Position Evaluation Uncertainty via Depth Variance
**STA561D: Probabilistic Machine Learning**

**Hypothesis:** Stockfish evaluation at shallow search depth is noisy — the engine has not seen far enough ahead to resolve tactical complications. Evaluation variance across depths is a proxy for position ambiguity.

- **Low variance across depths** → position is tactically resolved quickly; shallow and deep agree
- **High variance across depths** → position is genuinely ambiguous; evaluation shifts as the engine looks deeper

We test this across three position types of increasing complexity and compute Shannon entropy of the evaluation distribution as a principled uncertainty measure.

In [ ]:
import sys, os, asyncio
import chess, chess.engine, chess.svg
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import SVG, display
from dotenv import load_dotenv

load_dotenv()

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

STOCKFISH_PATH = os.getenv('STOCKFISH_PATH')
print(f'Stockfish path: {STOCKFISH_PATH}')

In [ ]:
# Three positions: quiet opening, tactical middlegame, sharp endgame
POSITIONS = {
    'Quiet Opening': {
        'fen': 'r1bqkb1r/pppp1ppp/2n2n2/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4',
        'description': 'Italian Game — symmetrical, low tension, evaluation should be stable across depths'
    },
    'Tactical Middlegame': {
        'fen': 'r1b1k2r/ppppqppp/2n2n2/2b1p3/2B1P3/2NP1N2/PPP2PPP/R1BQK2R w KQkq - 0 7',
        'description': 'Open middlegame — active pieces, potential tactics, moderate evaluation uncertainty'
    },
    'Sharp Tactical': {
        'fen': 'r2qkb1r/ppp2ppp/2np1n2/4p3/2BPP1b1/2N2N2/PPP2PPP/R1BQK2R w KQkq - 0 7',
        'description': 'Fried Liver-adjacent — sharp tactics, pins, uncastled king. High evaluation volatility expected'
    }
}

DEPTHS = [3, 5, 7, 9, 11, 13, 15, 18, 20]

for name, pos in POSITIONS.items():
    b = chess.Board(pos['fen'])
    print(f"--- {name} ---")
    print(f"    {pos['description']}")
    display(SVG(chess.svg.board(b, size=220)))

In [ ]:
def evaluate_at_depths(fen, depths, stockfish_path):
    """
    Run Stockfish at each specified depth and return centipawn evaluations.
    Returns list of (depth, centipawn_score) tuples.
    Mate scores are capped at ±1000 centipawns for comparability.
    """
    board = chess.Board(fen)
    evaluations = []

    with chess.engine.SimpleEngine.popen_uci(stockfish_path) as engine:
        for depth in depths:
            info = engine.analyse(board, chess.engine.Limit(depth=depth))
            score = info['score'].white()

            if score.is_mate():
                mate = score.mate()
                cp = 1000 if mate and mate > 0 else -1000
            else:
                cp = score.score()
                cp = max(-1000, min(1000, cp)) if cp is not None else 0

            evaluations.append((depth, cp / 100))  # convert to pawn units

    return evaluations


def discretize_for_entropy(evals, bins=10):
    """
    Bin evaluations and compute Shannon entropy.
    Higher entropy = more spread in evaluations = more uncertainty.
    """
    counts, _ = np.histogram(evals, bins=bins, range=(-10, 10))
    probs = counts / counts.sum()
    probs = probs[probs > 0]  # remove zero bins
    return -np.sum(probs * np.log2(probs))


print('Running depth-variance analysis...')
print(f'Depths tested: {DEPTHS}')
print()

results = {}
for pos_name, pos in POSITIONS.items():
    print(f'Evaluating: {pos_name}...', end=' ', flush=True)
    evals = evaluate_at_depths(pos['fen'], DEPTHS, STOCKFISH_PATH)
    results[pos_name] = evals
    scores = [e[1] for e in evals]
    print(f'done | range: [{min(scores):.2f}, {max(scores):.2f}] | std: {np.std(scores):.3f}')

print('\nAll evaluations complete.')

In [ ]:
# Compute summary statistics
summary_rows = []
for pos_name, evals in results.items():
    scores = [e[1] for e in evals]
    summary_rows.append({
        'Position':      pos_name,
        'Min Eval':      round(min(scores), 3),
        'Max Eval':      round(max(scores), 3),
        'Final Eval':    round(scores[-1], 3),
        'Std Dev':       round(np.std(scores), 3),
        'Range':         round(max(scores) - min(scores), 3),
        'Shannon Entropy': round(discretize_for_entropy(scores), 3),
    })

df_summary = pd.DataFrame(summary_rows)
print('EVALUATION UNCERTAINTY SUMMARY')
print('='*70)
print('(Higher std dev and entropy = more positional uncertainty)')
print()
print(df_summary.to_string(index=False))
print()
print('Interpretation:')
for _, row in df_summary.iterrows():
    if row['Std Dev'] < 0.1:
        label = 'Low uncertainty — evaluation stable across depths'
    elif row['Std Dev'] < 0.3:
        label = 'Moderate uncertainty — some depth-dependent volatility'
    else:
        label = 'High uncertainty — evaluation shifts significantly with depth'
    print(f"  {row['Position']}: {label}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Position Evaluation Uncertainty Across Search Depths\n'
             '(Stockfish centipawn score, White perspective)',
             fontsize=13, fontweight='bold')

pos_colors = {
    'Quiet Opening':      '#2ecc71',
    'Tactical Middlegame': '#e67e22',
    'Sharp Tactical':     '#e74c3c'
}

for i, (pos_name, evals) in enumerate(results.items()):
    depths_list = [e[0] for e in evals]
    scores_list = [e[1] for e in evals]
    std = np.std(scores_list)
    color = pos_colors[pos_name]
    ax = axes[i]

    ax.plot(depths_list, scores_list,
            marker='o', linewidth=2.5, markersize=7, color=color)
    ax.axhline(y=scores_list[-1], color='gray',
               linestyle='--', alpha=0.5, label=f'Final eval: {scores_list[-1]:+.2f}')

    # Shade uncertainty band ±1 std around mean
    mean_val = np.mean(scores_list)
    ax.axhspan(mean_val - std, mean_val + std,
               alpha=0.15, color=color, label=f'±1σ band (σ={std:.3f})')

    ax.set_title(f'{pos_name}\nStd Dev: {std:.3f}', fontweight='bold')
    ax.set_xlabel('Search Depth', fontsize=11)
    ax.set_ylabel('Evaluation (pawns)' if i == 0 else '', fontsize=11)
    ax.set_xticks(depths_list)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('uncertainty_depth_variance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uncertainty_depth_variance.png')

In [ ]:
# Chart 2: Uncertainty metrics comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Positional Uncertainty Metrics by Position Type', fontsize=13, fontweight='bold')

pos_names = df_summary['Position'].tolist()
colors = [pos_colors[p] for p in pos_names]

# Std Dev
bars1 = axes[0].bar(pos_names, df_summary['Std Dev'], color=colors, width=0.5)
axes[0].set_title('Standard Deviation of Evaluation\nAcross Depths', fontweight='bold')
axes[0].set_ylabel('Std Dev (pawns)')
axes[0].set_xticks(range(len(pos_names)))
axes[0].set_xticklabels(pos_names, rotation=15, ha='right')
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars1, df_summary['Std Dev']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', fontweight='bold')

# Shannon Entropy
bars2 = axes[1].bar(pos_names, df_summary['Shannon Entropy'], color=colors, width=0.5)
axes[1].set_title('Shannon Entropy of Evaluation Distribution\n(Higher = more uncertain)', fontweight='bold')
axes[1].set_ylabel('Entropy (bits)')
axes[1].set_xticks(range(len(pos_names)))
axes[1].set_xticklabels(pos_names, rotation=15, ha='right')
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(bars2, df_summary['Shannon Entropy']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('uncertainty_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uncertainty_metrics.png')

In [ ]:
# Convergence analysis: at what depth does evaluation stabilise?
print('CONVERGENCE ANALYSIS')
print('='*70)
CONVERGENCE_THRESHOLD = 0.10  # stabilised within 0.10 pawns of final value

for pos_name, evals in results.items():
    scores = [e[1] for e in evals]
    final = scores[-1]
    for i, (depth, score) in enumerate(evals):
        if abs(score - final) <= CONVERGENCE_THRESHOLD:
            print(f"  {pos_name}: converges at depth {depth} "
                  f"(eval={score:+.2f}, final={final:+.2f})")
            break
    else:
        print(f"  {pos_name}: does not converge within tested depths")

print()
print('KEY FINDING:')
stds = [(pos, np.std([e[1] for e in evals])) for pos, evals in results.items()]
stds.sort(key=lambda x: x[1], reverse=True)
print(f'Highest uncertainty: {stds[0][0]} (σ={stds[0][1]:.3f})')
print(f'Lowest uncertainty:  {stds[-1][0]} (σ={stds[-1][1]:.3f})')
print()
print('IMPLICATION FOR CHESS TUTOR:')
print('  Positions with high evaluation uncertainty (high σ) are inherently harder')
print('  to explain — the "correct" assessment depends on how deeply you calculate.')
print('  The system could flag high-variance positions to the learner as tactically')
print('  complex, adjusting explanation depth expectations accordingly.')